# Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas
# DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits as follows:
## • Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account", "general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g. if d%3 gives "account", write a question like “how do I update my registered mobile number”).
## • # Example roll number ...23 -> digits 2, 3
## • # digit 2 -> category[2 % 3] = general
## • # digit 3 -> category[3 % 3] = billing
## Output: Print your final 6-row DataFrame.
#### {
#### import pandas as pd
#### fixed_entries = [
#### {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
####  {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
#### {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
#### {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
#### ]}

In [1]:
import pandas as pd

fixed_entries = [{
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing",
    },{
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account",
    },{
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general",
    },{
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing",
    },
]

roll_number = input("Enter your roll number: ")
d1, d2 = int(roll_number[-2]), int(roll_number[-1])

categories = ["billing", "account", "general"]
cat1 = categories[d1 % 3]
cat2 = categories[d2 % 3]

faq_samples = {
    "billing": [
        {
            "question": "how do i download my payment receipt",
            "answer": "Go to Payments > Transaction History to download receipts.",
            "keywords": "receipt invoice download bill",
        },
        {
            "question": "what is the refund policy",
            "answer": "Refund requests must be submitted within 7 days of payment.",
            "keywords": "refund return money back",
        },
    ],
    "account": [
        {
            "question": "how do i update my registered mobile number",
            "answer": "Go to Profile > Contact Details and verify with an OTP.",
            "keywords": "update phone mobile number",
        },
        {
            "question": "how do i change my registered email address",
            "answer": "Submit a change request under Profile Settings with verification.",
            "keywords": "email change address profile",
        },
    ],
    "general": [
        {
            "question": "where is the campus library located",
            "answer": "The library is on the second floor of the main academic block.",
            "keywords": "library campus books location",
        },
        {
            "question": "how can i contact customer support",
            "answer": "Reach out via email at support@college.edu or call the helpline.",
            "keywords": "contact support helpline email",
        },
    ],
}

all_entries = fixed_entries + [faq_samples[cat1], faq_samples[cat2]]

df = pd.DataFrame(all_entries)
print("\nFinal Knowledge Base DataFrame: ")
print(df)

Enter your roll number:  1024170265


AttributeError: 'list' object has no attribute 'keys'

# Q2: Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence

In [ ]:
def score_hypothesis(query, df):
    q_words = set(query.lower().split())

    df["confidence"] = df.apply(lambda row: len(q_words & set(f"{row['question']} {row['keywords']}".lower().split())) / len(q_words), axis=1)
    return df[df["confidence"]>0].sort_values(by="confidence", ascending=False)

query = "how to pay fee"
results = score_hypothesis(query, df)
print(results[["question", "answer", "confidence"]])

# Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.

In [12]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]

target_category = cat1
result = same_category(target_category, df)

print("Questions under '{target_category}' category:")
print(result.to_string(index=False))

Questions under '{target_category}' category:
what is the annual fee
 how can i pay the fee
how can i get a refund


# Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named "your_roll_number"_faq_data.csv.

In [15]:
new_kw = input("Enter a new keyword to add: ")

df.at[0, "keywords"] = df.at[0, "keywords"] + " " + new_kw

filename = f"{roll_number}_faq_data.csv"
df.to_csv(filename, index = False)

print(f"\nUpdated Entry: \n{df.loc[0]}")
print(f"\nSaved Successfully to '{filename}'!")

Enter a new keyword to add:  1024170265



Updated Entry: 
question                           what is the annual fee
answer                          The annual fee is Rs 500.
keywords      fee cost price charge 1024170265 1024170265
category                                          billing
confidence                                           0.25
Name: 0, dtype: object

Saved Successfully to '1024170265_faq_data.csv'!


# Q5: Using groupby, print how many FAQ entries you have per category.

In [16]:
category_counts = df.groupby("category").size()

print("Number ofFAQ entries per category:")
print(category_counts)

Number ofFAQ entries per category:
category
account    1
billing    3
general    2
dtype: int64


# Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [17]:
def score_with_ties(query, df):
    q_words = set(query.lower().split())

    scores = df.apply(lambda r: len(q_words & set(f"{r['question']} {r['keywords']}".lower().split())), axis = 1,)
    max_score = scores.max()

    if max_score == 0:
        print("No matches found.\n")
        return 
    top_entries = df[scores == max_score]

    if len(top_entries) > 1:
        print(f"Tie Found ({len(top_entries)} matches): ")
    else:
        print("Single Best Match: ")

    print(top_entries[["question", "answer"]], "\n")

print("---- Test 1 (Tie) ----")
score_with_ties("fee", df)

print("---- Test 2 (No Tie) ----")
score_with_ties("password", df)

---- Test 1 (Tie) ----
Tie Found (2 matches): 
                 question                                      answer
0  what is the annual fee                   The annual fee is Rs 500.
3   how can i pay the fee  You can pay via UPI, card, or net banking. 

---- Test 2 (No Tie) ----
Single Best Match: 
                question                            answer
1  how to reset password  Go to Settings > Reset Password. 

